# Kaggle inference: B2 donor rerun 66 + B2-rand × 3 seed

Add Input: grace_b2_devign_donor_rerun_66_20261102.zip, grace_b2_rand_inputs.zip và model GGUF Q4_K_M. Bật GPU T4 và Internet. Không cần username Kaggle. Một deadline 9 giờ chung (chừa 5 phút), bắt đầu ở cell inference; cài đặt ở ngoài deadline. Thứ tự: 28 B2 validation → ghép 2.704 kết quả cũ và khóa ngưỡng trên đủ 2.732 → 38 B2 test → B2-rand validation ba seed → khóa ngưỡng → B2-rand test ba seed. Checkpoint chung lưu hai arm riêng; phiên tiếp tục có deadline mới. Một mẫu đang chạy có thể vượt hạn.


In [1]:
from pathlib import Path
import sys, os, json, subprocess, traceback, shutil, platform, importlib.metadata, zipfile, hashlib, time, re
from datetime import datetime, timezone

DIAG = Path('/kaggle/working/grace_diagnostics')
DIAG.mkdir(parents=True, exist_ok=True)
EVENTS = DIAG / 'events.log'

def note(message):
    text = str(message)
    print(text, flush=True)
    with EVENTS.open('a', encoding='utf-8') as f:
        f.write(f'[{datetime.now(timezone.utc).isoformat()}] {text}\n')

def diagnostics_zip():
    archive = shutil.make_archive('/kaggle/working/grace_diagnostics', 'zip', root_dir=DIAG)
    print('Tải log chẩn đoán từ Kaggle Output:', archive, flush=True)
    return archive

def run_logged(cmd, name, env=None):
    log = DIAG / f'{name}.log'
    note(f'Bắt đầu {name}: {cmd!r} | log đầy đủ: {log}')
    is_inference = name == 'inference_batch'
    with log.open('w', encoding='utf-8') as f:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, encoding='utf-8', errors='replace', bufsize=1, env=env)
        for line in proc.stdout:
            f.write(line); f.flush()
            if re.search(r'\b(?:Found|Skip(?:ping)?) link\b', line, flags=re.IGNORECASE):
                continue
            if is_inference:
                match = re.search(r'^(\S+).*\((\d+)/(\d+)\)\s*$', line.strip())
                if match:
                    run_key, done, total = match.group(1), int(match.group(2)), int(match.group(3))
                    if done % 100 == 0 or done == total:
                        print(f'Inference {run_key}: {done}/{total}', flush=True)
                elif line.strip():
                    print(line, end='', flush=True)
            else:
                print(line, end='', flush=True)
        code = proc.wait()
    if is_inference and '--status' in cmd:
        status_path = Path(cmd[cmd.index('--status') + 1])
        if status_path.is_file():
            status = json.loads(status_path.read_text(encoding='utf-8'))
            print(f"Inference summary: {status.get('completed_records_total', 0)} records; state={status.get('state')}", flush=True)
            for run_key, progress in status.get('runs', {}).items():
                print(f"  {run_key}: {progress.get('completed', 0)}/{progress.get('count', '?')} [{progress.get('state', 'pending')}]")
    note(f'{name} kết thúc với mã {code}')
    if code:
        raise subprocess.CalledProcessError(code, cmd)

def versions():
    result = {}
    for name in ('llama-cpp-python', 'transformers', 'huggingface_hub', 'scikit-learn', 'numpy', 'scipy'):
        try: result[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError: result[name] = 'not installed'
    note('Installed package versions: ' + json.dumps(result))
    return result

def capture_failure(shell, etype, evalue, tb, tb_offset=None):
    trace = ''.join(traceback.format_exception(etype, evalue, tb))
    try:
        with (DIAG / 'exceptions.log').open('a', encoding='utf-8') as f:
            f.write(f'\n[{datetime.now(timezone.utc).isoformat()}]\n{trace}\n')
        note(f'Lỗi: {etype.__name__}: {evalue}')
        versions()
        diagnostics_zip()
    except Exception as diagnostic_error:
        print('Không thể ghi đầy đủ chẩn đoán:', diagnostic_error, flush=True)
    shell.showtraceback((etype, evalue, tb), tb_offset=tb_offset)
    return []

get_ipython().set_custom_exc((Exception,), capture_failure)
note('UTC: ' + datetime.now(timezone.utc).isoformat())
note('Python: ' + sys.version)
note('Executable: ' + sys.executable)
note('Platform: ' + platform.platform())
for loc in ('/kaggle/working', '/kaggle/input'):
    try:
        usage = shutil.disk_usage(loc)
        note(f'Disk {loc}: total={usage.total} free={usage.free} bytes')
    except OSError as exc: note(f'Disk {loc}: {exc}')
versions()
for name, cmd in [('gpu', ['nvidia-smi']), ('cuda_compiler', ['nvcc', '--version']), ('cmake', ['cmake', '--version']), ('pip', [sys.executable, '-m', 'pip', '--version'])]:
    try: run_logged(cmd, name)
    except (OSError, subprocess.CalledProcessError) as exc: note(f'{name}: {type(exc).__name__}: {exc}')
diagnostics_zip()


UTC: 2026-11-01T23:41:34.523486+00:00
Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Executable: /usr/bin/python3
Platform: Linux-6.18.55+-x86_64-with-glibc2.39
Disk /kaggle/working: total=20957446144 free=20940386304 bytes
Disk /kaggle/input: total=20957446144 free=20940386304 bytes
Installed package versions: {"llama-cpp-python": "not installed", "transformers": "5.16.1", "huggingface_hub": "1.29.0", "scikit-learn": "1.6.1", "numpy": "2.1.3", "scipy": "1.16.3"}
Bắt đầu gpu: ['nvidia-smi'] | log đầy đủ: /kaggle/working/grace_diagnostics/gpu.log
Sun Nov  1 23:41:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usag

'/kaggle/working/grace_diagnostics.zip'

## Chọn bundle và cấu hình chạy


In [2]:
# Gắn đúng Kaggle Dataset bundle và Notebook Output chứa model.
KAGGLE_INPUT_ROOT = Path('/kaggle/input')
BUNDLE_WORK = Path('/kaggle/working/grace_b2_rand_bundles')
MODEL_FILENAME = 'Qwen3-Coder-30B-A3B-Instruct-Q4_K_M.gguf'
MODEL_SHA256 = 'f6050b4da22869aa3fb9d94bb3314e9fc5a027d4741c1f97761c387d6b2b2335'
SEEDS = (42, 43, 44)
RESUME_FROM_CHECKPOINT = False
CHECKPOINT_INPUT_PATH = None  # None: tự tìm grace_b2_rand_checkpoint.zip trong Input
if RESUME_FROM_CHECKPOINT:
    if CHECKPOINT_INPUT_PATH is None:
        candidates = sorted(KAGGLE_INPUT_ROOT.rglob('grace_b2_rand_checkpoint.zip'))
        if len(candidates) != 1: raise ValueError(f'Cần đúng một checkpoint: {candidates}')
        CHECKPOINT_INPUT_PATH = candidates[0]
    CHECKPOINT_INPUT_PATH = Path(CHECKPOINT_INPUT_PATH)
    if not CHECKPOINT_INPUT_PATH.is_file(): raise FileNotFoundError(CHECKPOINT_INPUT_PATH)
MODEL_REVISION = None  # None: khóa revision Hub tại thời điểm tải
LIMIT = None           # Chạy toàn bộ từng split
MAX_RUNTIME_HOURS = 9
CHECKPOINT_RESERVE_SECONDS = 300  # Dừng tạo record mới trước hạn để đóng gói checkpoint
GPU_LAYERS = -1
OUT_ROOT = Path('/kaggle/working/grace_b2_rand_three_seed_11h')
EXPECTED_RUNS = [('devign', 'validation'), ('reveal', 'validation'),
                 ('devign', 'test'), ('reveal', 'test')]


In [3]:
# Mã verifier và khóa ngưỡng được đóng gói trong notebook, không thực thi code từ ZIP Input.
(Path('/kaggle/working')/'donor_bundle_verify.py').write_text("\"\"\"Verify the portable B2 donor rerun bundle; standard library only, no LLM.\"\"\"\nimport hashlib\nimport json\nfrom collections import Counter\nfrom pathlib import Path\nimport zipfile\n\n\ndef parse_rows(data):\n    return [json.loads(line) for line in data.decode('utf-8').splitlines() if line.strip()]\n\n\ndef load_bundle(archive):\n    with zipfile.ZipFile(archive) as z:\n        if z.testzip() is not None:\n            raise ValueError('ZIP CRC check failed')\n        if len(z.namelist()) != len(set(z.namelist())):\n            raise ValueError('Duplicate ZIP members')\n        m = json.loads(z.read('bundle_manifest.json'))\n        if m['schema'] != 'grace_donor_rerun_bundle_v1' or m['count'] != 66:\n            raise ValueError('Wrong bundle schema or size')\n        if m['arm'] != 'B2' or m['dataset'] != 'devign' or not m['is_partial_split']:\n            raise ValueError('Wrong experiment identity')\n        for name, expected in m['files_sha256'].items():\n            if hashlib.sha256(z.read(name)).hexdigest() != expected:\n                raise ValueError(f'File hash mismatch: {name}')\n        if m['prompts_sha256'] != m['files_sha256']['prompts.jsonl']:\n            raise ValueError('Prompt manifest hashes disagree')\n        rows = parse_rows(z.read('prompts.jsonl'))\n        if Counter(r['split'] for r in rows) != Counter(validation=28, test=38):\n            raise ValueError('Wrong split counts')\n        if len(rows) != 66 or len({r['id'] for r in rows}) != 66:\n            raise ValueError('Wrong count or duplicate ID')\n        audit = parse_rows(z.read('rerun_audit.jsonl'))\n        by_id = {r['id']: r for r in audit}\n        if len(audit) != 66 or len(by_id) != 66 or set(by_id) != {r['id'] for r in rows}:\n            raise ValueError('Audit IDs differ from rerun prompts')\n        for r in rows:\n            if r['arm'] != 'B2' or r['dataset'] != 'devign':\n                raise ValueError('Wrong row identity')\n            if r['donor_snapshot_sha256'] != m['donor_snapshot_sha256']:\n                raise ValueError('Wrong donor snapshot')\n            if hashlib.sha256(r['prompt'].encode('utf-8')).hexdigest() != r['prompt_sha256']:\n                raise ValueError('Prompt hash mismatch')\n            a = by_id[r['id']]\n            if a['new_prompt_sha256'] != r['prompt_sha256'] or a['old_prompt_sha256'] == r['prompt_sha256']:\n                raise ValueError('A rerun row does not contain the changed prompt')\n            if a['split'] != r['split'] or a['model_sha256'] != m['expected_inference_identity']['model_sha256']:\n                raise ValueError('Audit split/model mismatch')\n            if r['tokens']['prompt_tokens'] + 64 > m['prompt_budget_tokens']:\n                raise ValueError('Prompt exceeds context budget')\n        for split in ('validation', 'test'):\n            split_rows = parse_rows(z.read(f'splits/devign/{split}/prompts.jsonl'))\n            if split_rows != [r for r in rows if r['split'] == split]:\n                raise ValueError('Combined and split prompt files disagree')\n        expected = parse_rows(z.read('validation_support/expected_full_validation.jsonl'))\n        reusable = parse_rows(z.read('validation_support/reusable_records.jsonl'))\n        expected_by_id = {r['id']: r for r in expected}\n        if len(expected) != 2732 or len(expected_by_id) != 2732:\n            raise ValueError('Wrong full validation size')\n        new_validation = [r for r in rows if r['split'] == 'validation']\n        if len(reusable) != 2704 or len({r['id'] for r in reusable}) != 2704:\n            raise ValueError('Wrong reusable validation size')\n        if {r['id'] for r in reusable} & {r['id'] for r in new_validation}:\n            raise ValueError('Old scores overlap changed validation prompts')\n        if {r['id'] for r in reusable + new_validation} != set(expected_by_id):\n            raise ValueError('Validation merge would not cover the full set')\n        for r in reusable + new_validation:\n            e = expected_by_id[r['id']]\n            if (r['split'], r['dataset'], r['label'], r['prompt_sha256']) != ('validation', 'devign', e['label'], e['prompt_sha256']):\n                raise ValueError('Validation ID/label/hash mismatch')\n        for r in reusable:\n            if r['arm'] != 'B2' or r['seed'] != 42 or r['model_sha256'] != m['expected_inference_identity']['model_sha256']:\n                raise ValueError('Reusable score inference identity mismatch')\n    return m, rows\n\n\nif __name__ == '__main__':\n    import argparse\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument('bundle', type=Path)\n    args = parser.parse_args()\n    manifest, prompts = load_bundle(args.bundle)\n    print(json.dumps({'bundle': str(args.bundle), 'verified': True,\n                      'prompts': len(prompts), 'counts': dict(Counter(r['split'] for r in prompts)),\n                      'reusable_validation_scores': 2704}, indent=2))\n",encoding='utf-8')
(Path('/kaggle/working')/'donor_validation_lock.py').write_text("\"\"\"CPU-only, validation-only threshold locking. Portable into Kaggle notebooks.\"\"\"\nimport hashlib\nimport json\nimport math\nimport os\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nRULES={'version':'validation_mcc_grid_v1','grid':[i/100 for i in range(1,100)],\n       'objective':'MCC on finite valid validation scores only','comparison':'score >= threshold',\n       'tie_break':'within 1e-9: closest to 0.5, then lower threshold',\n       'degenerate':'if every MCC is undefined, lock 0.5 with MCC=null; report explicitly',\n       'coverage':'require every validation ID; invalid rows retained in coverage but excluded from fitting',\n       'test_usage':'never read test records or metrics to choose threshold'}\ndef digest_json(value):\n    return hashlib.sha256(json.dumps(value,sort_keys=True,separators=(',',':'),ensure_ascii=False).encode()).hexdigest()\nRULES_SHA256=digest_json(RULES)\n\ndef file_sha256(path):\n    h=hashlib.sha256()\n    with Path(path).open('rb') as f:\n        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)\n    return h.hexdigest()\n\ndef read_jsonl(path):\n    with Path(path).open(encoding='utf-8') as f:\n        return [json.loads(l) for l in f if l.strip()]\n\ndef write_json(path,value):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    tmp=path.with_suffix(path.suffix+'.tmp')\n    with tmp.open('w',encoding='utf-8',newline='\\n') as f:\n        json.dump(value,f,ensure_ascii=False,indent=2,allow_nan=False);f.write('\\n');f.flush();os.fsync(f.fileno())\n    os.replace(tmp,path)\n\ndef mcc(y,p):\n    tp=sum(a==1 and b==1 for a,b in zip(y,p));tn=sum(a==0 and b==0 for a,b in zip(y,p))\n    fp=sum(a==0 and b==1 for a,b in zip(y,p));fn=sum(a==1 and b==0 for a,b in zip(y,p))\n    den=math.sqrt((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn))\n    return (tp*tn-fp*fn)/den if den else None\n\ndef pick_threshold(rows):\n    valid=[]\n    for r in rows:\n        value=r.get('s')\n        if r.get('valid') and isinstance(value,(int,float)) and math.isfinite(value):\n            if not 0<=value<=1:raise ValueError('Score outside [0,1]')\n            valid.append(r)\n    if not valid:raise ValueError('No finite valid validation scores; cannot lock threshold')\n    if {r['label'] for r in valid}!={0,1}:raise ValueError('Validation fitting needs both classes among valid scores')\n    y=[r['label'] for r in valid];scores=[r['s'] for r in valid]\n    best_t=.5;best_m=None;grid=[]\n    for t in RULES['grid']:\n        value=mcc(y,[int(v>=t) for v in scores]);grid.append({'threshold':t,'mcc':value})\n        if value is None:continue\n        if best_m is None or value>best_m+1e-9 or (abs(value-best_m)<=1e-9 and (abs(t-.5),t)<(abs(best_t-.5),best_t)):\n            best_t,best_m=t,value\n    return {'tau':best_t,'validation_mcc':best_m,'n_valid':len(valid),'n_invalid':len(rows)-len(valid),\n            'score_coverage':len(valid)/len(rows),'degenerate':best_m is None,'grid_results':grid}\n\ndef inference_identity(manifest):\n    return {'model_sha256':manifest['model_sha256'],'tokenizer':manifest['tokenizer'],\n            'tokenizer_revision':manifest['tokenizer_revision'],'quantization':manifest['quantization'],\n            'backend':'llama-cpp-python','backend_version':manifest['llama_cpp_python'],\n            'scoring':manifest['scoring'],'seed_recorded':42,'context_tokens':manifest['context_tokens']}\n\ndef validate_validation(expected,records,arm,dataset,model_sha256):\n    exp={str(r['id']):r for r in expected}\n    if len(exp)!=len(expected):raise ValueError('Duplicate expected validation IDs')\n    seen={}\n    for r in records:\n        uid=str(r['id'])\n        if r.get('split')!='validation' or r.get('dataset')!=dataset or r.get('arm')!=arm:\n            raise ValueError('Only the requested arm/dataset validation records may be fitted')\n        if uid in seen:raise ValueError('Duplicate validation records')\n        e=exp.get(uid)\n        if e is None or r.get('label')!=e['label'] or r.get('prompt_sha256')!=e['prompt_sha256']:\n            raise ValueError('Validation ID/label/prompt identity mismatch')\n        if r.get('model_sha256')!=model_sha256 or r.get('seed')!=42:\n            raise ValueError('Validation model/seed mismatch')\n        seen[uid]=r\n    return exp,seen\n\ndef make_lock(expected,records,arm,dataset,run_manifest,validation_prompt_set_sha256,source_evidence=None):\n    exp,seen=validate_validation(expected,records,arm,dataset,run_manifest['model_sha256'])\n    if set(exp)!=set(seen):raise ValueError(f'Incomplete validation: {len(seen)}/{len(exp)}; do not fit on partial data')\n    ordered=[seen[uid] for uid in sorted(seen)]\n    result=pick_threshold(ordered)\n    evidence=[{k:r.get(k) for k in ('id','label','prompt_sha256','model_sha256','seed','s','valid')} for r in ordered]\n    # Normalize invalid scores for strict JSON; their raw validity remains recorded.\n    for r in evidence:\n        if r['s'] is not None and not math.isfinite(r['s']):r['s']=None\n    return {'arm':arm,'dataset':dataset,'split_used':'validation','locked_at_utc':datetime.now(timezone.utc).isoformat(),\n            'rules_sha256':RULES_SHA256,'identity':inference_identity(run_manifest),\n            'validation_prompt_set_sha256':validation_prompt_set_sha256,'n_expected':len(exp),\n            'validation_evidence_sha256':digest_json(evidence),'source_evidence':source_evidence or [],\n            'temporal_claim':'locked at actual time; does not certify that external test inference had not started',**result}\n\ndef persist_lock(path,entry):\n    path=Path(path)\n    data=json.loads(path.read_text(encoding='utf-8')) if path.exists() else {'schema':'thresholds_v1','rules':RULES,'rules_sha256':RULES_SHA256,'entries':{}}\n    if data.get('rules_sha256')!=RULES_SHA256:raise ValueError('Threshold rules differ; create a new version')\n    key=f\"{entry['arm']}/{entry['dataset']}\";old=data['entries'].get(key)\n    if old:\n        for k in ('rules_sha256','identity','validation_prompt_set_sha256','validation_evidence_sha256','tau'):\n            if old.get(k)!=entry.get(k):raise ValueError(f'Locked threshold is immutable; conflicting {k} for {key}')\n        return old\n    data['entries'][key]=entry;write_json(path,data);return entry\n\ndef require_test_lock(path,arm,dataset,run_manifest,validation_prompt_set_sha256=None):\n    path=Path(path)\n    if not path.exists():raise ValueError('Test blocked: thresholds.json missing; complete validation first')\n    data=json.loads(path.read_text(encoding='utf-8'));entry=data.get('entries',{}).get(f'{arm}/{dataset}')\n    if not entry:raise ValueError(f'Test blocked: no locked threshold for {arm}/{dataset}')\n    if data.get('rules_sha256')!=RULES_SHA256 or entry.get('rules_sha256')!=RULES_SHA256:\n        raise ValueError('Test blocked: different threshold selection rules')\n    if entry.get('identity')!=inference_identity(run_manifest):raise ValueError('Test blocked: model/tokenizer/backend differs from validation')\n    if validation_prompt_set_sha256 and entry.get('validation_prompt_set_sha256')!=validation_prompt_set_sha256:\n        raise ValueError('Test blocked: validation prompt set changed')\n    return entry\n\ndef evaluate_at_locked_threshold(records,entry):\n    valid=[r for r in records if r.get('valid') and isinstance(r.get('s'),(int,float)) and math.isfinite(r['s'])]\n    y=[r['label'] for r in valid];pred=[int(r['s']>=entry['tau']) for r in valid]\n    tp=sum(a==1 and b==1 for a,b in zip(y,pred));tn=sum(a==0 and b==0 for a,b in zip(y,pred))\n    fp=sum(a==0 and b==1 for a,b in zip(y,pred));fn=sum(a==1 and b==0 for a,b in zip(y,pred))\n    return {'tau':entry['tau'],'n':len(valid),'score_coverage':len(valid)/len(records) if records else 0,\n            'mcc':mcc(y,pred),'accuracy':(tp+tn)/len(y) if y else None,\n            'confusion_matrix':[[tn,fp],[fn,tp]],'threshold_locked_at_utc':entry['locked_at_utc'],\n            'threshold_validation_evidence_sha256':entry['validation_evidence_sha256']}\n",encoding='utf-8')
sys.path.insert(0, '/kaggle/working')
import donor_bundle_verify, donor_validation_lock

# Đọc đúng ZIP tổng hoặc 12 ZIP split; xác minh arm, seed, hash, ID và số mẫu.
import io
BUNDLE_WORK.mkdir(parents=True, exist_ok=True)
found = {}
def register_bundle(manifest_bytes, prompt_bytes, origin):
    manifest = json.loads(manifest_bytes)
    if manifest.get('arm') != 'B2-rand': return
    key = (manifest.get('seed'), manifest.get('dataset'), manifest.get('split'))
    if key[0] not in SEEDS or key[1:] not in EXPECTED_RUNS: return
    digest = hashlib.sha256(prompt_bytes).hexdigest()
    if digest != manifest['prompts_sha256']: raise ValueError(f'Hash sai: {origin}')
    rows = [json.loads(x) for x in prompt_bytes.decode('utf-8').splitlines() if x.strip()]
    if len(rows) != manifest['count'] or len({str(x['id']) for x in rows}) != len(rows): raise ValueError(f'Count/ID sai: {origin}')
    if any((x.get('seed'),x.get('dataset'),x.get('split'),x.get('arm')) != (*key,'B2-rand') or hashlib.sha256(x['prompt'].encode()).hexdigest()!=x['prompt_sha256'] for x in rows): raise ValueError(f'Identity/prompt hash sai: {origin}')
    if key in found and found[key]['digest'] != digest: raise ValueError(f'Bundle trùng khác nội dung: {key}')
    found[key] = dict(manifest=manifest_bytes,prompts=prompt_bytes,digest=digest,origin=str(origin))
for path in sorted(KAGGLE_INPUT_ROOT.rglob('bundle_manifest.json')):
    prompts = path.with_name('prompts.jsonl')
    if prompts.is_file(): register_bundle(path.read_bytes(),prompts.read_bytes(),path.parent)
archives = sorted(set(KAGGLE_INPUT_ROOT.rglob('grace_b2_rand_inputs.zip')) | set(KAGGLE_INPUT_ROOT.rglob('*_b2_rand_seed_*_kaggle.zip')))
for path in archives:
    with zipfile.ZipFile(path) as z:
        names=set(z.namelist())
        if {'bundle_manifest.json','prompts.jsonl'} <= names: register_bundle(z.read('bundle_manifest.json'),z.read('prompts.jsonl'),path)
        for member in sorted(x for x in names if '_b2_rand_seed_' in x and x.endswith('_kaggle.zip')):
            with zipfile.ZipFile(io.BytesIO(z.read(member))) as inner:
                register_bundle(inner.read('bundle_manifest.json'),inner.read('prompts.jsonl'),f'{path}!{member}')
expected=[(seed,ds,sp) for seed in SEEDS for ds,sp in EXPECTED_RUNS]
missing=[key for key in expected if key not in found]
if missing: raise FileNotFoundError(f'Thiếu bundle: {missing}. Add Input grace_b2_rand_inputs.zip')
BUNDLE_PATHS={}
for key in expected:
    seed,ds,sp=key; item=found[key]; dest=BUNDLE_WORK/f'{seed}_{ds}_{sp}.zip'
    with zipfile.ZipFile(dest,'w',compression=zipfile.ZIP_DEFLATED) as z:
        z.writestr('bundle_manifest.json',item['manifest']); z.writestr('prompts.jsonl',item['prompts'])
    BUNDLE_PATHS[key]=dest
    note(f'Bundle {key}: {json.loads(item["manifest"])["count"]} mẫu, sha256={item["digest"]}, source={item["origin"]}')
diagnostics_zip()

# Kaggle có thể giữ ZIP hoặc tự giải nén Dataset thành thư mục.
RERUN_BUNDLE_ID = 'grace_b2_devign_donor_rerun_66_20261102'
rerun_zip_paths = sorted(p for p in KAGGLE_INPUT_ROOT.rglob(RERUN_BUNDLE_ID + '.zip') if p.is_file())
rerun_dirs = []
for path in KAGGLE_INPUT_ROOT.rglob('bundle_manifest.json'):
    try:
        manifest = json.loads(path.read_text(encoding='utf-8'))
    except (ValueError, OSError):
        continue
    if manifest.get('schema') == 'grace_donor_rerun_bundle_v1' and manifest.get('bundle_id') == RERUN_BUNDLE_ID:
        rerun_dirs.append(path.parent)
if not rerun_zip_paths and not rerun_dirs:
    raise FileNotFoundError('Không tìm thấy donor rerun trong Input (ZIP hoặc thư mục đã giải nén). Hãy Add Input Dataset donor rerun.')

# Dựng ZIP tạm từ danh sách file trong manifest, sau đó dùng cùng verifier như ZIP gốc.
rerun_candidates = [(path, str(path)) for path in rerun_zip_paths]
for i, folder in enumerate(sorted(rerun_dirs)):
    manifest_path = folder / 'bundle_manifest.json'
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    tmp_zip = BUNDLE_WORK / f'donor_rerun_unpacked_{i}.zip'
    folder_root = folder.resolve()
    with zipfile.ZipFile(tmp_zip, 'w', compression=zipfile.ZIP_STORED) as archive:
        archive.writestr('bundle_manifest.json', manifest_path.read_bytes())
        for name in sorted(manifest['files_sha256']):
            file_path = (folder / name).resolve()
            if not file_path.is_relative_to(folder_root) or not file_path.is_file():
                raise ValueError(f'File bundle thiếu hoặc đường dẫn không hợp lệ: {folder}/{name}')
            archive.write(file_path, name)
    rerun_candidates.append((tmp_zip, str(folder)))

verified = []
for path, origin in rerun_candidates:
    manifest, rows = donor_bundle_verify.load_bundle(path)
    if manifest.get('bundle_id') != RERUN_BUNDLE_ID: raise ValueError(f'Sai bundle ID: {origin}')
    verified.append((path, origin, manifest, rows))
RERUN_INPUT, RERUN_ORIGIN, RERUN_MANIFEST, rerun_rows = verified[0]
if any(item[2] != RERUN_MANIFEST or item[3] != rerun_rows for item in verified[1:]):
    raise ValueError(f'Có nhiều bundle donor rerun khác nhau trong Input: {[item[1] for item in verified]}')
if RERUN_MANIFEST['model_sha256'] != MODEL_SHA256: raise ValueError('Model rerun khác contract')
if RERUN_MANIFEST['expected_inference_identity']['backend_version'] != '0.3.16': raise ValueError('Backend rerun khác contract')
RERUN_BUNDLES = {}
with zipfile.ZipFile(RERUN_INPUT) as z:
    RERUN_REUSABLE = donor_bundle_verify.parse_rows(z.read('validation_support/reusable_records.jsonl'))
    RERUN_EXPECTED = donor_bundle_verify.parse_rows(z.read('validation_support/expected_full_validation.jsonl'))
    RERUN_ORIGINAL_RUN = json.loads(z.read('validation_support/original_run_manifest.json'))
    if donor_validation_lock.inference_identity(RERUN_ORIGINAL_RUN) != RERUN_MANIFEST['expected_inference_identity']:
        raise ValueError('Identity kết quả validation cũ không khớp bundle')
    for split in ('validation','test'):
        raw = z.read(f'splits/devign/{split}/prompts.jsonl')
        manifest = dict(RERUN_MANIFEST, split=split, count=RERUN_MANIFEST['counts_by_split'][split], prompts_sha256=hashlib.sha256(raw).hexdigest())
        dest=BUNDLE_WORK/f'donor_rerun_devign_{split}.zip'
        with zipfile.ZipFile(dest,'w',compression=zipfile.ZIP_DEFLATED) as out:
            out.writestr('bundle_manifest.json',json.dumps(manifest)); out.writestr('prompts.jsonl',raw)
        RERUN_BUNDLES[split]=dest
note(f'Đã kiểm tra donor rerun: {RERUN_ORIGIN}; validation=28, test=38; 2704 score validation tái sử dụng')


Bundle (42, 'devign', 'validation'): 2732 mẫu, sha256=8112a6478f8eecd8e7011af881ce9057d02caf42cb8367515108785ff55ad65a, source=/kaggle/input/datasets/duchuynguyen1802/grace-b2-rand-inputs-zip/seed_42/devign_validation_b2_rand_seed_42_kaggle
Bundle (42, 'reveal', 'validation'): 2273 mẫu, sha256=dd08b1ffd36690bf6f2ab00f8fda1708d6b88e6fe9957893519075d35d06d8a3, source=/kaggle/input/datasets/duchuynguyen1802/grace-b2-rand-inputs-zip/seed_42/reveal_validation_b2_rand_seed_42_kaggle
Bundle (42, 'devign', 'test'): 2732 mẫu, sha256=237a138fc911224fe8d0059bc101971f33d8a1e182e7be3d911535d2a3fcb7cb, source=/kaggle/input/datasets/duchuynguyen1802/grace-b2-rand-inputs-zip/seed_42/devign_test_b2_rand_seed_42_kaggle
Bundle (42, 'reveal', 'test'): 2274 mẫu, sha256=39ccb2c61797c272a4dcca0419887587e3197bcc7869497ad6a52e8d19c93a66, source=/kaggle/input/datasets/duchuynguyen1802/grace-b2-rand-inputs-zip/seed_42/reveal_test_b2_rand_seed_42_kaggle
Bundle (43, 'devign', 'validation'): 2732 mẫu, sha256=1e9b6f

ValueError: Cần đúng một ZIP donor rerun trong Input: []

## Cài thư viện và xác minh CUDA


In [ ]:
# Cài từng nhóm để nhận diện chính xác bước thất bại; giữ nguyên phiên bản baseline.
# T4 dùng SM75; Kaggle đặt libcuda.so trong /usr/local/nvidia/lib64 nên thêm thư mục này cho CMake.
os.environ['CMAKE_ARGS'] = '-DGGML_CUDA=on -DCMAKE_CUDA_ARCHITECTURES=75 -DCMAKE_LIBRARY_PATH=/usr/local/nvidia/lib64'
os.environ['CMAKE_BUILD_PARALLEL_LEVEL'] = '2'
os.environ['FORCE_CMAKE'] = '1'
note('Build config: CMAKE_ARGS=' + os.environ['CMAKE_ARGS'] + '; CMAKE_BUILD_PARALLEL_LEVEL=' + os.environ['CMAKE_BUILD_PARALLEL_LEVEL'] + '; FORCE_CMAKE=' + os.environ['FORCE_CMAKE'])
run_logged([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', 'transformers==4.57.1', 'huggingface_hub', 'scikit-learn==1.5.2'], 'install_dependencies')
run_logged([sys.executable, '-m', 'pip', 'install', '-v', '--no-cache-dir', '--no-binary', 'llama-cpp-python', 'llama-cpp-python==0.3.16'], 'install_llama_cuda')
versions()
diagnostics_zip()


In [ ]:
import llama_cpp
versions()
note('llama-cpp-python GPU offload: ' + str(llama_cpp.llama_supports_gpu_offload()))
assert llama_cpp.llama_supports_gpu_offload(), 'llama-cpp-python chưa có CUDA; xem install_llama_cuda.log'
diagnostics_zip()


## Tải model và chạy inference


In [ ]:
# Tìm đúng tên file model trong toàn bộ Input đã gắn, bất kể username/slug.
model_files = sorted(path for path in KAGGLE_INPUT_ROOT.rglob(MODEL_FILENAME) if path.is_file())
if len(model_files) != 1:
    raise FileNotFoundError(f'Cần đúng một {MODEL_FILENAME} trong Kaggle Input; '
                            f'tìm thấy: {list(map(str, model_files))}. '
                            'Hãy Add Input chứa model và bỏ bản trùng nếu có; '
                            'notebook không tự tải lại GGUF từ Hugging Face.')
model_path = str(model_files[0])
revision = 'local-input'
note(f'Dùng model input: {model_path}')
print('Model input:', model_path)
print('Batch output directory:', OUT_ROOT)


In [ ]:
%%writefile /kaggle/working/kaggle_inference.py
import argparse
import hashlib
import json
import math
import os
import platform
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import numpy as np


def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def write_json_atomic(path, payload):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(payload, indent=2), encoding='utf-8')
    os.replace(tmp, path)


def binary_scores(logits, label_ids):
    values = np.asarray(logits, dtype=np.float64).reshape(-1)
    if not np.all(np.isfinite(values)):
        raise ValueError('Non-finite logits')
    maximum = np.max(values)
    normalizer = float(maximum + np.log(np.exp(values - maximum).sum()))
    lp0 = float(values[label_ids[0]] - normalizer)
    lp1 = float(values[label_ids[1]] - normalizer)
    score = float(np.exp(lp1 - np.logaddexp(lp0, lp1)))
    best = int(np.argmax(values))
    top = np.argsort(values)[-5:][::-1]
    return {'logprob_0': lp0, 'logprob_1': lp1, 'p0': math.exp(lp0), 'p1': math.exp(lp1),
            's': score, 'pred_argmax': int(score >= 0.5), 'greedy_token_id': best,
            'top5': [{'token_id': int(i), 'logprob': float(values[i] - normalizer)} for i in top]}


def load_bundle(bundle):
    with zipfile.ZipFile(bundle) as z:
        manifest = json.loads(z.read('bundle_manifest.json'))
        prompt_bytes = z.read('prompts.jsonl')
    if hashlib.sha256(prompt_bytes).hexdigest() != manifest['prompts_sha256']:
        raise ValueError(f'Prompt bundle hash mismatch: {bundle}')
    rows = [json.loads(line) for line in prompt_bytes.decode('utf-8').splitlines() if line.strip()]
    if len(rows) != manifest['count'] or len({str(r['id']) for r in rows}) != len(rows):
        raise ValueError(f'Prompt count or unique ID check failed: {bundle}')
    return manifest, rows


def evaluate_output(output):
    from sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score,
                                 confusion_matrix, f1_score, matthews_corrcoef, precision_score,
                                 recall_score, roc_auc_score)
    records = [json.loads(line) for line in output.read_text(encoding='utf-8').splitlines() if line.strip()]
    actual = [r for r in records if r['response_valid']]
    truth = [r['label'] for r in actual]
    predicted = [int(r['raw']) for r in actual]
    metrics = {'n_total': len(records), 'n_score_valid': sum(bool(r['valid']) for r in records),
               'n_response_valid': len(actual),
               'response_coverage': len(actual) / len(records) if records else None,
               'accuracy': None, 'balanced_accuracy': None, 'precision': None, 'recall': None,
               'f1': None, 'mcc': None, 'confusion_matrix': None, 'auroc': None, 'auprc': None}
    if actual:
        metrics.update(accuracy=accuracy_score(truth, predicted),
                       balanced_accuracy=balanced_accuracy_score(truth, predicted),
                       precision=precision_score(truth, predicted, zero_division=0),
                       recall=recall_score(truth, predicted, zero_division=0),
                       f1=f1_score(truth, predicted, zero_division=0),
                       mcc=matthews_corrcoef(truth, predicted),
                       confusion_matrix=confusion_matrix(truth, predicted, labels=[0, 1]).tolist())
    score_rows = [r for r in records if r['valid']]
    if score_rows and len({r['label'] for r in score_rows}) == 2:
        labels = [r['label'] for r in score_rows]
        scores = [r['s'] for r in score_rows]
        metrics.update(auroc=roc_auc_score(labels, scores),
                       auprc=average_precision_score(labels, scores))
    return metrics


def run_batch(runs, model_path, model_revision, gpu_layers, limit, deadline_epoch,
              checkpoint_reserve_seconds, status_path, seed, expected_model_sha256, arm):
    from llama_cpp import Llama, __version__ as llama_version
    from transformers import AutoTokenizer
    if llama_version != '0.3.16': raise ValueError(f'Backend version mismatch: {llama_version}')

    started = time.time()
    stop_at = deadline_epoch - checkpoint_reserve_seconds
    status_path = Path(status_path)
    status_path.parent.mkdir(parents=True, exist_ok=True)
    status = {'state': 'starting', 'timed_out': False,
              'started_at': datetime.fromtimestamp(started, timezone.utc).isoformat(),
              'deadline_at': datetime.fromtimestamp(deadline_epoch, timezone.utc).isoformat(),
              'stop_new_records_at': datetime.fromtimestamp(stop_at, timezone.utc).isoformat(),
              'max_runtime_hours': (deadline_epoch - started) / 3600,
              'checkpoint_reserve_seconds': checkpoint_reserve_seconds,
              'runs': {}, 'completed_records_total': 0}
    for dataset, split, bundle, out in runs:
        key = f'{dataset}/{split}'
        status['runs'][key] = {'state': 'pending', 'bundle': bundle, 'output': out}
    write_json_atomic(status_path, status)

    def stop_for_deadline(current=None):
        status['state'] = 'timed_out'
        status['timed_out'] = True
        status['timed_out_at'] = datetime.now(timezone.utc).isoformat()
        status['stopped_after_run'] = current
        write_json_atomic(status_path, status)

    if time.time() >= stop_at:
        stop_for_deadline(None)
        return

    model_path = Path(model_path)
    model_digest = sha256(model_path)
    if model_digest != expected_model_sha256: raise ValueError('Model SHA256 không khớp contract')
    if model_revision == 'local-input':
        model_revision = f'local-sha256:{model_digest}'
    first_manifest, _ = load_bundle(runs[0][2])
    for dataset, split, bundle, out in runs:
        current_manifest, _ = load_bundle(bundle)
        if (current_manifest['tokenizer'], current_manifest['tokenizer_revision'],
            current_manifest['prompt_budget_tokens']) != (first_manifest['tokenizer'],
            first_manifest['tokenizer_revision'], first_manifest['prompt_budget_tokens']):
            raise ValueError('All four bundles must use the same tokenizer revision and context budget')
    tokenizer = AutoTokenizer.from_pretrained(first_manifest['tokenizer'],
                                                revision=first_manifest['tokenizer_revision'])
    llm = Llama(model_path=str(model_path), n_ctx=first_manifest['prompt_budget_tokens'], n_batch=256,
                n_gpu_layers=gpu_layers, logits_all=True, verbose=True)
    if time.time() >= stop_at:
        stop_for_deadline(None)
        return
    label_ids = []
    for label in ('0', '1'):
        hf_ids = tokenizer.encode(label, add_special_tokens=False)
        gguf_ids = llm.tokenize(label.encode(), add_bos=False, special=False)
        if len(hf_ids) != 1 or hf_ids != gguf_ids:
            raise ValueError(f'Label token mismatch for {label}: HF={hf_ids}, GGUF={gguf_ids}')
        label_ids.append(hf_ids[0])
    if label_ids[0] == label_ids[1]:
        raise ValueError('Label token IDs must differ')

    status['state'] = 'running'
    write_json_atomic(status_path, status)
    for dataset, split, bundle, out_text in runs:
        key = f'{dataset}/{split}'
        out = Path(out_text)
        out.mkdir(parents=True, exist_ok=True)
        manifest, rows = load_bundle(bundle)
        if (manifest.get('arm'), manifest.get('seed'), manifest['dataset'], manifest['split']) != (arm, seed, dataset, split):
            raise ValueError(f'Bundle identity mismatch for {key}: {manifest["dataset"]}/{manifest["split"]}')
        if limit is not None:
            rows = rows[:limit]
        identity = {'bundle_sha256': sha256(bundle),
                    'prompt_set_sha256': manifest['prompts_sha256'], 'model_sha256': model_digest,
                    'model_repo': manifest['model_repo'], 'quantization': manifest['quantization'],
                    'model_file': model_path.name, 'llama_cpp_python': llama_version,
                    'model_repo_revision': model_revision, 'tokenizer': manifest['tokenizer'],
                    'tokenizer_revision': manifest['tokenizer_revision'], 'gpu_layers': gpu_layers,
                    'context_tokens': manifest['prompt_budget_tokens'],
                    'scoring': 'unconstrained_first_token_logits_v1', 'limit': limit,
                    'python': platform.python_version(), 'dataset': dataset, 'split': split, 'arm': arm, 'prompt_seed': seed}
        identity_file = out / 'run_manifest.json'
        previous_identity = (json.loads(identity_file.read_text(encoding='utf-8'))
                             if identity_file.exists() else None)
        if previous_identity is not None:
            # Ignore only archive-byte hashes: ZIP timestamps/compression metadata may change.
            # Prompt identity is verified below against every checkpoint row's prompt_sha256.
            ignored = {'bundle_sha256', 'prompt_set_sha256'}
            previous_stable = {k: v for k, v in previous_identity.items() if k not in ignored}
            current_stable = {k: v for k, v in identity.items() if k not in ignored}
            if previous_stable != current_stable:
                raise ValueError(f'Existing output has different model/run settings for {key}; use a new OUT_ROOT')
            if ('prompt_set_sha256' in previous_identity and
                    previous_identity['prompt_set_sha256'] != identity['prompt_set_sha256']):
                raise ValueError(f'Prompt bundle content changed for {key}; refusing to resume')
        output = out / 'records.jsonl'
        done = {}
        if output.exists():
            for line in output.read_text(encoding='utf-8').splitlines():
                if line.strip():
                    row = json.loads(line)
                    row_id = str(row['id'])
                    if row_id in done:
                        raise ValueError(f'Duplicate checkpoint ID in {key}: {row_id}')
                    done[row_id] = row
        expected_ids = {str(item['id']) for item in rows}
        unexpected_ids = set(done) - expected_ids
        if unexpected_ids:
            raise ValueError(f'Checkpoint contains IDs absent from current bundle {key}: {sorted(unexpected_ids)[:10]}')
        for item in rows:
            row_id = str(item['id'])
            if row_id in done:
                if done[row_id].get('prompt_sha256') != item['prompt_sha256']:
                    raise ValueError(f'Checkpoint prompt mismatch: {key}/{row_id}')
                if done[row_id].get('model_sha256') != model_digest:
                    raise ValueError(f'Checkpoint model hash mismatch: {key}/{row_id}')
        # Persist the refreshed archive hash only after checkpoint IDs and prompt hashes pass.
        identity_file.write_text(json.dumps(identity, indent=2), encoding='utf-8')
        progress = status['runs'][key]
        progress.update(state='running', count=len(rows), completed=len(done),
                        records_path=str(output), bundle_sha256=identity['bundle_sha256'])
        status['completed_records_total'] = sum(x.get('completed', 0) for x in status['runs'].values())
        write_json_atomic(status_path, status)
        if len(done) > len(rows):
            raise ValueError(f'Checkpoint has more rows than this bundle: {key}')
        print(f'{key}: verified checkpoint IDs={len(done)}; skipping these; remaining={len(rows) - len(done)}', flush=True)

        with output.open('a', encoding='utf-8', newline='\n') as f:
            for item in rows:
                row_id = str(item['id'])
                if row_id in done:
                    if done[row_id]['prompt_sha256'] != item['prompt_sha256']:
                        raise ValueError(f'Checkpoint prompt mismatch: {key}/{row_id}')
                    continue
                if time.time() >= stop_at:
                    progress.update(state='checkpointed', completed=len(done))
                    status['completed_records_total'] = sum(x.get('completed', 0) for x in status['runs'].values())
                    stop_for_deadline(key)
                    return
                prompt = item['prompt']
                if hashlib.sha256(prompt.encode()).hexdigest() != item['prompt_sha256']:
                    raise ValueError(f'Prompt hash mismatch: {key}/{row_id}')
                messages = [{'role': 'user', 'content': prompt}]
                chat = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
                hf_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True)
                gguf_ids = llm.tokenize(chat.encode('utf-8'), add_bos=False, special=True)
                if list(hf_ids) != gguf_ids:
                    raise ValueError(f'Tokenizer mismatch at {key}/{row_id}; abort rather than alter prompt')
                if len(gguf_ids) > manifest['prompt_budget_tokens']:
                    raise ValueError(f'Token budget exceeded at {key}/{row_id}')
                start = time.monotonic()
                llm.reset()
                llm.eval(gguf_ids)
                scores = binary_scores(llm.scores[llm.n_tokens - 1], label_ids)
                raw = llm.detokenize([scores['greedy_token_id']]).decode('utf-8', errors='replace').strip()
                row = {k: item[k] for k in ('id', 'dataset', 'split', 'label', 'prompt_sha256', 'demo_ids', 'tokens', 'graph_missing')}
                row.update(scores, arm=arm, raw=raw, response_valid=raw in ('0', '1'), valid=True,
                           token_ids={'0': label_ids[0], '1': label_ids[1]}, model_sha256=model_digest,
                           backend='llama-cpp-python', quantization='Q4_K_M', model_repo=manifest['model_repo'],
                           model_repo_revision=model_revision, seed=seed, timestamp=datetime.now(timezone.utc).isoformat(),
                           logprob_distribution='unconstrained_first_token', elapsed_s=time.monotonic()-start)
                f.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + '\n')
                f.flush()
                os.fsync(f.fileno())
                done[row_id] = row
                progress['completed'] = len(done)
                status['completed_records_total'] = sum(x.get('completed', 0) for x in status['runs'].values())
                status['last_checkpoint_at'] = datetime.now(timezone.utc).isoformat()
                write_json_atomic(status_path, status)
                print(f'{key} {row_id}: s={scores["s"]:.4f}, raw={raw!r} ({len(done)}/{len(rows)})', flush=True)
                if time.time() >= stop_at:
                    progress['state'] = 'checkpointed'
                    stop_for_deadline(key)
                    return

        metrics = evaluate_output(output)
        (out / 'evaluation.json').write_text(json.dumps(metrics, indent=2, allow_nan=False), encoding='utf-8')
        progress.update(state='complete', completed=len(rows), evaluation=str(out / 'evaluation.json'))
        status['completed_records_total'] = sum(x.get('completed', 0) for x in status['runs'].values())
        write_json_atomic(status_path, status)

    status['state'] = 'complete'
    status['timed_out'] = False
    status['completed_at'] = datetime.now(timezone.utc).isoformat()
    write_json_atomic(status_path, status)


if __name__ == '__main__':
    ap = argparse.ArgumentParser()
    ap.add_argument('--model', required=True)
    ap.add_argument('--seed', type=int, required=True)
    ap.add_argument('--arm', choices=['B2', 'B2-rand'], required=True)
    ap.add_argument('--expected-model-sha256', required=True)
    ap.add_argument('--model-revision', required=True)
    ap.add_argument('--gpu-layers', type=int, default=-1)
    ap.add_argument('--limit', type=int)
    ap.add_argument('--deadline-epoch', type=float, required=True)
    ap.add_argument('--checkpoint-reserve-seconds', type=int, default=300)
    ap.add_argument('--status', required=True)
    ap.add_argument('--run', action='append', nargs=4, metavar=('DATASET', 'SPLIT', 'BUNDLE', 'OUT'), required=True)
    args = ap.parse_args()
    run_batch(args.run, args.model, args.model_revision, args.gpu_layers, args.limit,
              args.deadline_epoch, args.checkpoint_reserve_seconds, args.status, args.seed, args.expected_model_sha256, args.arm)


In [ ]:
# Hạn chung cho mọi seed và phase; chạy validation của cả ba seed trước test.
OUT_ROOT.mkdir(parents=True, exist_ok=True)
if RESUME_FROM_CHECKPOINT and not any(OUT_ROOT.iterdir()):
    with zipfile.ZipFile(CHECKPOINT_INPUT_PATH) as z:
        root=OUT_ROOT.resolve()
        for member in z.infolist():
            if not (OUT_ROOT/member.filename).resolve().is_relative_to(root): raise ValueError('Unsafe checkpoint path')
        z.extractall(OUT_ROOT)
    note(f'Restored checkpoint: {CHECKPOINT_INPUT_PATH}')
RUN_DEADLINE_EPOCH = time.time() + MAX_RUNTIME_HOURS * 3600
timed_out=False
def lock_validation_thresholds():
    # Grid và tie-break theo protocol.analysis.json; tuyệt đối không đọc test.
    import math
    for seed in SEEDS:
        for dataset in ('devign','reveal'):
            key=(seed,dataset,'validation')
            manifest=json.loads(found[key]['manifest']); rows=[json.loads(x) for x in (OUT_ROOT/f'seed_{seed}'/dataset/'validation'/'records.jsonl').read_text(encoding='utf-8').splitlines() if x.strip()]
            if len(rows)!=manifest['count'] or len({str(x['id']) for x in rows})!=manifest['count']:
                raise ValueError(f'Validation chưa đủ: {key}')
            valid=[x for x in rows if x.get('valid') and math.isfinite(x.get('s',float('nan')))]
            scored=[]
            for i in range(1,100):
                threshold=i/100; tp=sum(x['label']==1 and x['s']>=threshold for x in valid); tn=sum(x['label']==0 and x['s']<threshold for x in valid)
                fp=sum(x['label']==0 and x['s']>=threshold for x in valid); fn=sum(x['label']==1 and x['s']<threshold for x in valid)
                den=(tp+fp)*(tp+fn)*(tn+fp)*(tn+fn)
                if den: scored.append((threshold,(tp*tn-fp*fn)/math.sqrt(den)))
            if scored:
                best=max(v for _,v in scored); tied=[(t,v) for t,v in scored if best-v<=1e-9]
                threshold,mcc=min(tied,key=lambda x:(abs(x[0]-0.5),x[0]))
            else: threshold,mcc=0.5,None
            target=OUT_ROOT/f'seed_{seed}'/dataset/'validation'/'threshold_lock.json'
            payload={'protocol':'validation_mcc_grid_v1','seed':seed,'dataset':dataset,'threshold':threshold,'mcc':mcc,'valid':len(valid),'total':len(rows),'validation_records_sha256':hashlib.sha256((target.parent/'records.jsonl').read_bytes()).hexdigest(),'prompt_set_sha256':manifest['prompts_sha256']}
            target.write_text(json.dumps(payload,indent=2),encoding='utf-8')
            note(f'Locked validation threshold {dataset} seed={seed}: {threshold}, MCC={mcc}')

# Chạy B2 donor rerun trước; khóa validation đầy đủ trước test.
if LIMIT is not None:
    raise ValueError('LIMIT phải là None để giữ contract ghép validation đầy đủ')
RERUN_ROOT=OUT_ROOT/'b2_donor_rerun_66'
RERUN_LOCK=RERUN_ROOT/'thresholds.json'
def run_donor_split(split):
    status_path=RERUN_ROOT/f'{split}_status.json'
    cmd=[sys.executable,'-u','/kaggle/working/kaggle_inference.py','--model',model_path,
         '--model-revision',revision,'--seed','42','--arm','B2','--expected-model-sha256',MODEL_SHA256,
         '--gpu-layers',str(GPU_LAYERS),'--deadline-epoch',str(RUN_DEADLINE_EPOCH),
         '--checkpoint-reserve-seconds',str(CHECKPOINT_RESERVE_SECONDS),'--status',str(status_path),
         '--run','devign',split,str(RERUN_BUNDLES[split]),str(RERUN_ROOT/'devign'/split)]
    run_logged(cmd,f'donor_rerun_{split}')
    return json.loads(status_path.read_text(encoding='utf-8'))

if time.time() >= RUN_DEADLINE_EPOCH-CHECKPOINT_RESERVE_SECONDS:
    timed_out=True
else:
    status=run_donor_split('validation')
    timed_out=bool(status.get('timed_out'))
if not timed_out:
    validation_dir=RERUN_ROOT/'devign'/'validation'
    run_manifest=json.loads((validation_dir/'run_manifest.json').read_text(encoding='utf-8'))
    if donor_validation_lock.inference_identity(run_manifest) != RERUN_MANIFEST['expected_inference_identity']:
        raise ValueError('Runtime rerun không khớp inference identity của score cũ')
    new_rows=donor_validation_lock.read_jsonl(validation_dir/'records.jsonl')
    if len(new_rows)!=28: raise ValueError('Phải có đủ 28 score validation mới')
    entry=donor_validation_lock.make_lock(
        RERUN_EXPECTED,RERUN_REUSABLE+new_rows,'B2','devign',run_manifest,
        RERUN_MANIFEST['full_validation_prompt_set_sha256'],
        source_evidence=[{'bundle_id':RERUN_MANIFEST['bundle_id'],
                          'reusable_records_sha256':RERUN_MANIFEST['files_sha256']['validation_support/reusable_records.jsonl'],
                          'rerun_validation_records_sha256':donor_validation_lock.file_sha256(validation_dir/'records.jsonl')}])
    locked=donor_validation_lock.persist_lock(RERUN_LOCK,entry)
    donor_validation_lock.require_test_lock(RERUN_LOCK,'B2','devign',run_manifest,RERUN_MANIFEST['full_validation_prompt_set_sha256'])
    note(f"Đã khóa ngưỡng B2 donor mới từ {locked['n_expected']} validation: tau={locked['tau']}")
    if time.time() >= RUN_DEADLINE_EPOCH-CHECKPOINT_RESERVE_SECONDS:
        timed_out=True
    else:
        status=run_donor_split('test')
        timed_out=bool(status.get('timed_out'))

for phase in (() if timed_out else ('validation','test')):
    if phase == 'test':
        if LIMIT is not None: raise ValueError('LIMIT chỉ dùng kiểm tra; không chạy test xác nhận với LIMIT')
        lock_validation_thresholds()
    for seed in SEEDS:
        if time.time() >= RUN_DEADLINE_EPOCH-CHECKPOINT_RESERVE_SECONDS:
            timed_out=True; break
        status_path=OUT_ROOT/f'seed_{seed}'/f'{phase}_status.json'
        cmd=[sys.executable,'-u','/kaggle/working/kaggle_inference.py','--model',model_path,
             '--model-revision',revision,'--seed',str(seed),'--arm','B2-rand','--expected-model-sha256',MODEL_SHA256,
             '--gpu-layers',str(GPU_LAYERS),'--deadline-epoch',str(RUN_DEADLINE_EPOCH),
             '--checkpoint-reserve-seconds',str(CHECKPOINT_RESERVE_SECONDS),'--status',str(status_path)]
        if LIMIT is not None: cmd += ['--limit',str(LIMIT)]
        for dataset in ('devign','reveal'):
            cmd += ['--run',dataset,phase,str(BUNDLE_PATHS[(seed,dataset,phase)]),str(OUT_ROOT/f'seed_{seed}'/dataset/phase)]
        run_logged(cmd, f'inference_seed_{seed}_{phase}')
        status=json.loads(status_path.read_text(encoding='utf-8'))
        if status.get('timed_out'): timed_out=True; break
    if timed_out: break
summary={'timed_out':timed_out,'deadline_epoch':RUN_DEADLINE_EPOCH,'seeds':list(SEEDS),'budget_hours':MAX_RUNTIME_HOURS}
(OUT_ROOT/'batch_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
archive=shutil.make_archive('/kaggle/working/grace_b2_rand_checkpoint' if timed_out else str(OUT_ROOT),'zip',root_dir=OUT_ROOT)
print('Checkpoint' if timed_out else 'Hoàn thành',archive,flush=True)
diagnostics_zip()


In [ ]:
print('Tải ZIP kết quả hoặc checkpoint từ Kaggle Output. B2 donor rerun nằm trong b2_donor_rerun_66; B2-rand nằm trong seed_*.')


Không tự chạy test xác nhận trước khi dữ liệu, arm, ngưỡng và protocol được khóa. Nếu model OOM, tokenizer không khớp hoặc thiếu GPU, dừng và lưu log; không thay model hoặc ràng buộc ngầm.
